This test is on
1. i.i.d. rayleigh fading channel
2. 4*4 MIMO channel
3. 1000 symbols per channel
4. total of 1000 channels

ver 11.24.04.35


In [1]:
"""
DONT RUN THIS CODE BLOCK WHEN USING LOCAL COMPUTER
THIS IS ONLY FOR GOOGLE COLAB
"""
!cd /content
!rm -rf EE_MIMO_PROJECT
!git clone https://github.com/billlllllllly/EE_MIMO_PROJECT.git
# load the whole repo
%cd EE_MIMO_PROJECT
%pip install -r requirements.txt
# change the dir
!ls
# just for checking that the current folder is correct

Cloning into 'EE_MIMO_PROJECT'...
remote: Enumerating objects: 422, done.
remote: Counting objects: 100% (422/422), done.
remote: Compressing objects: 100% (303/303), done.
remote: Total 422 (delta 183), reused 270 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (422/422), 7.99 MiB | 22.72 MiB/s, done.
Resolving deltas: 100% (183/183), done.
/content/EE_MIMO_PROJECT
NYCU_scene  requirements.txt  simulation_results  src


In [2]:
# import models
from src.models.MAP import MAP
from src.models.MMSE import MMSE
from src.models.SIC import SIC
from src.models.partialMAP import partialMAP

In [17]:
# mimo settings
import torch
from src.channel.rayleighfading import rayleighfading
BLOCKS = 1000
SYMBOLS_PER_CHANNEL = 10
# total of (SYMBOLS_PER_CHANNEL * BLOCKS) symbols are transmitted
NT = 2
NR = 2
SNR_DB = list(range(0, 21, 2))
MODULATION = "QPSK"

def generateChannel(NR, NT):
    return rayleighfading(NR=NR, NT=NT)

qpsk = torch.tensor([-1-1j, -1+1j, 1+1j, 1-1j], dtype=torch.complex128) / 2**0.5
symbol_qpsk = torch.tensor([[0, 0], [1, 0], [1, 1], [0, 1]])
bpsk = torch.tensor([-1, 1], dtype=torch.complex128) / 2**0.5
symbol_bpsk = torch.tensor([[0], [1]])

In [18]:
import inspect

print(inspect.getfile(partialMAP))
print(inspect.signature(partialMAP))

/content/EE_MIMO_PROJECT/src/models/partialMAP.py
(*, verbose=True, csv_log_dir=None)


In [ ]:
from datetime import datetime
from pathlib import Path
from tempfile import mkdtemp
from uuid import uuid4
from zoneinfo import ZoneInfo

from src.channel.testbench import Testbench


class PartialMAPBySNR(partialMAP):
    def updateNoiseVar(self, noiseVar):
        super().updateNoiseVar(noiseVar)
        self.csv_log_dir = self.log_dirs_by_variance[self.noiseVar]


# Use a fresh directory because Partial MAP appends rows to its CSV files.
pmap_log_dir = Path(mkdtemp(prefix="rayleigh_pmap_"))
pmap = PartialMAPBySNR(verbose=False)
models = {
    "MAP": MAP(),
    "MMSE": MMSE(),
    "QR-SIC": SIC(),
    "Partial MAP": pmap,
}

if MODULATION == "QPSK":
    SYMBOL = symbol_qpsk
    BITLABEL = qpsk
else:
    SYMBOL = symbol_bpsk
    BITLABEL = bpsk

bench = Testbench(models, BITLABEL, SYMBOL, SYMBOLS_PER_CHANNEL, SNR_DB)
# The testbench sets this variance before each detection batch.
pmap.log_dirs_by_variance = {
    variance: pmap_log_dir / f"{str(snr).removesuffix('.0')}dB"
    for snr, variance in zip(bench.snr_db, bench.noise_variances)
}
csi = torch.stack([generateChannel(NR=NR, NT=NT) for _ in range(BLOCKS)])
run_id = uuid4().hex
config_folder = f"rayleigh_{NR}_by_{NT}_{MODULATION}"
ERROR_CSV = pmap_log_dir / "rayleigh_bit_errors.csv"
cumulative = None
ber, bits_per_snr = bench.run(csi, csv_path=ERROR_CSV, plot_antenna_ber=True)
completed_at = datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds")

print(f"{bits_per_snr:,} bits per SNR")
print("SNR (dB)   " + "  ".join(f"{name:>12}" for name in models))
for snr, row in zip(SNR_DB, ber):
    print(f"{snr:8g}   " + "  ".join(f"{value:12.4g}" for value in row))

In [ ]:
# Save this run to Drive and refresh cumulative BER for each SNR and detector.
from google.colab import drive
from src.simulation.ber_history import save_ber_history

drive.mount("/content/drive")
out_dir = Path("/content/drive/MyDrive/EE_MIMO_PROJECT/02_colab_output") / config_folder
cumulative = save_ber_history(ERROR_CSV, out_dir, run_id, completed_at)
print(f"Saved layer CSVs, cumulative error counts, and simulation logs to {out_dir}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

snr_db = np.asarray(SNR_DB)
fig, ax = plt.subplots(figsize=(8, 5))

for j, name in enumerate(models):
    if cumulative is None:
        points = np.asarray(ber[:, j])
        totals = np.full(len(snr_db), bits_per_snr)
    else:
        rows = [cumulative[snr, name] for snr in snr_db]
        points = np.array([row["ber"] for row in rows])
        totals = np.array([row["total_bits_transmitted"] for row in rows])
    line, = ax.semilogy(
        snr_db, np.where(points > 0, points, np.nan), "o-", label=name
    )

    zero = points == 0
    if zero.any():
        ax.semilogy(
            snr_db[zero],
            1 / totals[zero],
            "v",
            color=line.get_color(),
        )

scope = "Cumulative BER" if cumulative is not None else "Run BER"
TITLE = f"{NT}by{NR}, {MODULATION}, Rayleigh, {scope}"
ax.set(xlabel="SNR (dB)", ylabel="BER",
       title=TITLE)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
print("Downward triangles: zero observed errors, shown at one error divided by the transmitted bit count.")

plt.savefig(f'{TITLE}.png', dpi=300, bbox_inches='tight')
try:
    from google.colab import files
except ImportError:
    pass
else:
    download_csv = ERROR_CSV if cumulative is None else ERROR_CSV.with_name("cumulative_rayleigh_bit_errors.csv")
    files.download(str(download_csv))
    files.download(f'{TITLE}.png')

plt.show()